# Классические NLP-задачи — версия без решений

Ниже 5 задач для самостоятельного разбора. Для каждого метода:

1. **Сначала реализуй алгоритм вручную** (на numpy) — чтобы понять, что он делает;
2. **Сравни** ручную реализацию с библиотечной (sklearn / gensim);
3. **Обучи классификатор** и посмотри метрики.


In [2]:
import math
import os
import random
import re
from typing import List

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity


Общая картина

В этом ноутбуке мы идём от самого простого представления текста к более сложным:

| Метод | Идея | Что сохраняет | Что теряет |
|-------|------|---------------|------------|
| **Bag of Words** | Считаем слова | Наличие слов | Порядок, редкость |
| **TF-IDF** | Считаем слова × их редкость | Наличие + важность слов | Порядок |
| **n-граммы** | Считаем пары/тройки соседних слов | Локальный порядок | Дальний контекст |
| **Word2Vec** | Учим плотные векторы слов по контексту | Семантическое сходство | Порядок в документе |
| **fastText** | Word2Vec + символьные подслова | Морфология, редкие слова | — |

Каждый следующий метод добавляет что-то, чего не хватало предыдущему.
BoW не видит разницы между редким и частым словом — TF-IDF это чинит.
TF-IDF не видит «отменить заказ» как единое целое — n-граммы это чинят.
n-граммы не знают, что «матч» и «чемпионат» близки по смыслу — Word2Vec это чинит.

## Задача 1: Bag of Words для SMS spam detection


Что такое Bag of Words

Bag of Words — самое простое представление текста в виде вектора.

Алгоритм:
1. Собираем **словарь** — множество всех уникальных слов в корпусе.
2. Каждый документ превращаем в вектор длины |словарь|.
3. В позиции $j$ стоит **количество вхождений** слова $j$ в документ.

$$\text{BoW}(d, w) = \text{count}(w, d)$$

Пример: словарь = `[bonus, call, free, now]`

| Текст | bonus | call | free | now |
|-------|-------|------|------|-----|
| `"free bonus free"` | 1 | 0 | 2 | 0 |
| `"call now"` | 0 | 1 | 0 | 1 |

Порядок слов полностью теряется — отсюда и название «мешок слов».
Но для спам-детекции порядок обычно не важен: достаточно знать, что
в сообщении есть `free`, `win`, `bonus` — и это уже сильный сигнал.

Почему `CountVectorizer` + `MultinomialNB`

**CountVectorizer** — sklearn-реализация Bag of Words. Строит словарь и возвращает sparse-матрицу счётчиков.

**MultinomialNB** — наивный байесовский классификатор. Для каждого класса запоминает профиль — как часто встречается каждое слово. Для нового документа выбирает класс, при котором вероятность увидеть такой набор слов максимальна:

$$P(\text{class} \mid \text{doc}) \propto P(\text{class}) \cdot \prod_{w \in \text{doc}} P(w \mid \text{class})$$

«Мультиномиальный» — потому что считает документ результатом мультиномиального распределения (бросаешь кубик со словами N раз). Поэтому ему нужны именно **целые счётчики** от CountVectorizer, а не вещественные TF-IDF веса.

Есть короткие SMS. Нужно определить `spam` или `ham`.

**Шаг 1** — реализуй Bag of Words вручную:
- Собери словарь (все уникальные слова, отсортированные)
- Для каждого документа посчитай количество вхождений каждого слова
- Результат: матрица (n_docs × n_vocab) целых чисел

**Шаг 2** — сравни с `CountVectorizer` и убедись, что матрицы совпадают.

**Шаг 3** — обучи `MultinomialNB` на обеих матрицах и сравни accuracy.


In [8]:
# TODO: сгенерируй sms_spam_small.csv, если файла нет
# Подсказка: отдельно собреи шаблоны для spam и ham

In [ ]:
# TODO: Шаг 1 — реализуй Bag of Words вручную
# Реализуй класс ManualBoW с методами fit_transform(texts) и transform(texts):
#   fit_transform: построить словарь, посчитать вхождения, вернуть матрицу
#   transform:     применить готовый словарь к новым текстам
#
# Подсказка: matrix[i, word_to_idx[word]] += 1

In [ ]:
# TODO: Шаг 2 — сравни ручной BoW со sklearn CountVectorizer
# Вызови CountVectorizer на тех же текстах и проверь:
#   assert np.max(np.abs(manual_bow_matrix - sklearn_bow_matrix)) == 0

In [ ]:
# TODO: Шаг 3 — обучи классификатор и сравни accuracy ручной vs sklearn
# TODO: загрузи sms_spam_small.csv, train_test_split с stratify
# TODO: обучи ручной ManualBoW: fit_transform на train, transform на test
# TODO: обучи sklearn CountVectorizer: fit_transform на train, transform на test
# TODO: обучи MultinomialNB на обеих матрицах
# TODO: выведи accuracy обеих моделей — они должны совпасть
# TODO: посмотри на самые частые токены


In [ ]:
# ---------- Проверка твоего решения (задача 1) ----------
# Ожидаемые переменные:
#   acc_bow       — accuracy на тесте
#   vectorizer_bow / vectorizer — обученный CountVectorizer
#   model_bow / model           — обученный MultinomialNB или LogisticRegression

try:
    _acc = acc_bow
except NameError:
    try:
        _acc = accuracy_score(y_test, model_bow.predict(X_test_bow))
    except NameError:
        raise AssertionError(
            "Не найдены переменные acc_bow или (model_bow, X_test_bow, y_test). "
            "Обучи модель и посчитай accuracy."
        )

assert _acc > 0.75, (
    f"Accuracy = {_acc:.4f} — слишком низкая. Возможные причины:\n"
    "  1) CountVectorizer обучен на тесте (fit_transform вместо transform)\n"
    "  2) Перепутаны метки spam/ham\n"
    "  3) Модель не обучена (забыл вызвать .fit())"
)

try:
    _v = vectorizer_bow if 'vectorizer_bow' in dir() else vectorizer
    assert hasattr(_v, 'vocabulary_'), "CountVectorizer не обучен."
    assert isinstance(_v, CountVectorizer), (
        f"Ожидался CountVectorizer, но получен {type(_v).__name__}. "
        "В задаче 2 нужен именно Bag of Words (CountVectorizer)."
    )
except NameError:
    pass

print(f"Задача 1: accuracy = {_acc:.4f} — проверка пройдена ✓")

### Если проверка не прошла — частые ошибки задачи 1

| Ошибка | Что происходит | Как исправить |
|--------|---------------|---------------|
| TfidfVectorizer + MultinomialNB | MultinomialNB ожидает неотрицательные счётчики, TF-IDF нарушает его предположения | Используй `CountVectorizer` для MultinomialNB |
| Нет `stratify` при split | Один класс может быть недопредставлен в тесте → нестабильные метрики | `train_test_split(..., stratify=y)` |
| `predict` на train данных | Оцениваешь на том, на чём обучал → завышенная accuracy | `model.predict(X_test_bow)` |
| `fit_transform` на тесте | Словарь перестраивается под тест → несовпадающие признаки | `vectorizer.transform(X_test)` (без fit!) |

## Задача 2: TF-IDF + Logistic Regression для тональности отзывов


Что такое TF-IDF и зачем он нужен после BoW

Проблема BoW: все слова одинаково важны. Слово «товар», которое встречается в каждом отзыве, весит столько же, сколько «ужасный», которое встречается только в негативных. TF-IDF это чинит.

**TF (Term Frequency)** — сколько раз слово встретилось в документе:
$$\text{TF}(t, d) = \text{count}(t, d)$$

**IDF (Inverse Document Frequency)** — насколько слово редкое в корпусе:
$$\text{IDF}(t) = \ln\!\left(\frac{1 + N}{1 + \text{df}(t)}\right) + 1$$

где $N$ — количество документов, $\text{df}(t)$ — в скольких документах встретилось слово $t$.

**TF-IDF = TF × IDF** — это и есть вся формула.

Но sklearn'овский `TfidfVectorizer` по умолчанию дополнительно делает **L2-нормализацию** каждой строки (`norm='l2'`). Это не часть определения TF-IDF, а практический трюк: без нормализации длинные документы получают бо́льшие значения просто из-за количества слов. L2-норма это выравнивает. Чтобы ручная реализация совпала со sklearn — нужно нормализовать.

Интуиция: слово, которое часто встречается в данном документе (высокий TF), но редко в корпусе (высокий IDF) — скорее всего важное для этого документа. А слово «и», которое есть везде, получит низкий IDF и будет подавлено.

Для тональности это критично: «отлично» и «ужасно» — редкие маркерные слова, а «товар» и «доставка» — общие для обоих классов.

Есть короткие отзывы о товаре. Нужно предсказать `positive` или `negative`.

**Шаг 1** — реализуй TF-IDF вручную:
- Построй словарь, посчитай TF-матрицу
- Вычисли IDF по формуле из теоретического блока выше
- Умножь TF × IDF и нормализуй строки по L2

**Шаг 2** — сравни с `TfidfVectorizer` и убедись, что матрицы совпадают.

**Шаг 3** — обучи `LogisticRegression` на обеих матрицах и сравни accuracy.


In [ ]:
# TODO: сгенерируй reviews_small.csv, если файла нет
# Подсказка: используй маленький словарь положительных и отрицательных слов


In [ ]:
# TODO: Шаг 1 — реализуй TF-IDF вручную
# Реализуй класс ManualTfidf с методами fit_transform(texts) и transform(texts):
#   fit_transform: построить словарь, посчитать TF, IDF, вернуть TF-IDF матрицу
#   transform:     применить готовый словарь и IDF к новым текстам
#
# Формулы:
#   TF[i, j] = количество вхождений слова j в документ i
#   IDF[j]   = ln((1 + N) / (1 + df[j])) + 1   (smooth IDF)
#   TF-IDF   = TF * IDF, затем L2-нормализация каждой строки
#
# Подсказка: используй numpy, не sklearn

In [ ]:
# TODO: Шаг 2 — сравни ручную реализацию со sklearn
# Вызови TfidfVectorizer на тех же текстах и проверь:
#   assert np.max(np.abs(manual_matrix - sklearn_matrix)) < 1e-10

In [ ]:
# TODO: Шаг 3 — обучи классификатор и сравни accuracy ручной vs sklearn
# TODO: загрузи reviews_small.csv, train_test_split с stratify
# TODO: обучи ручной ManualTfidf: fit_transform на train, transform на test
# TODO: обучи sklearn TfidfVectorizer: fit_transform на train, transform на test
# TODO: обучи LogisticRegression на обеих матрицах
# TODO: выведи accuracy обеих моделей — они должны совпасть
# TODO: посмотри на самые важные слова по коэффициентам модели


In [ ]:
# ---------- Проверка твоего решения (задача 2) ----------
# Этот блок ожидает, что ты создал переменные:
#   acc_tfidf     — accuracy на тесте
#   vectorizer    — обученный TfidfVectorizer (или vectorizer_tfidf)
#   X_train_tfidf — матрица признаков train
#   X_test_tfidf  — матрица признаков test
#   model         — обученная LogisticRegression (или model_tfidf)

try:
    _acc = acc_tfidf
except NameError:
    try:
        _acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    except NameError:
        raise AssertionError(
            "Не найдены переменные acc_tfidf или (model, X_test_tfidf, y_test). "
            "Убедись, что ты обучил модель и посчитал accuracy."
        )

assert _acc > 0.70, (
    f"Accuracy = {_acc:.4f} — слишком низкая. Возможные причины:\n"
    "  1) fit_transform вызван на тесте вместо transform\n"
    "  2) Vectorizer обучен на всех данных (data leakage)\n"
    "  3) Перепутаны X и y при обучении"
)

try:
    _vt = vectorizer_tfidf if 'vectorizer_tfidf' in dir() else vectorizer
    assert hasattr(_vt, 'vocabulary_'), "TfidfVectorizer не обучен — вызови fit_transform на train."
    _n_feat = len(_vt.vocabulary_)
    print(f"  Количество признаков: {_n_feat}")
except NameError:
    print("  (vectorizer не найден, проверка пропущена)")

print(f"Задача 2: accuracy = {_acc:.4f} — проверка пройдена ✓")

### Если проверка не прошла — частые ошибки задачи 2

| Ошибка | Что происходит | Как исправить |
|--------|---------------|---------------|
| `fit_transform` на тесте | Словарь перестраивается, признаки train/test не совпадают → accuracy ~50% | Используй `vectorizer.transform(X_test)` |
| Обучение vectorizer на всех данных до split | Data leakage: IDF посчитан с учётом теста → завышенные метрики | Сначала `train_test_split`, потом `fit_transform` только на train |
| Забыл `lowercase=True` | `Отличный` и `отличный` — разные признаки, словарь раздут | `TfidfVectorizer(lowercase=True)` (по умолчанию True) |
| `predict` на train вместо test | Оценка на тех же данных, на которых обучали → нереалистичная accuracy | `model.predict(X_test_tfidf)` |

## Задача 3: n-граммы поверх TF-IDF для intent classification


Что такое n-граммы и зачем они нужны

BoW и TF-IDF рассматривают слова по отдельности. Но иногда смысл — именно в сочетании:

- «заказ» → встречается в order, cancel, delivery
- «отменить» → встречается в cancel
- «отменить заказ» (биграмма) → встречается **только** в cancel

**Unigram** — одно слово: `хочу`, `отменить`, `заказ`.
**Bigram** — пара соседних слов: `хочу отменить`, `отменить заказ`.
**Trigram** — тройка: `хочу отменить заказ`.

n-граммы сохраняют **локальный порядок** слов. Это особенно полезно для коротких фраз, где каждое слово на счету.

На практике биграммы строятся поверх TF-IDF: сначала извлекаем все unigrams + bigrams, потом считаем для них TF-IDF как для обычных «слов». sklearn делает это одним параметром: `TfidfVectorizer(ngram_range=(1, 2))`.

Есть короткие запросы пользователей: `order`, `cancel`, `delivery`, `complaint`.

**Шаг 1** — реализуй извлечение n-грамм вручную:
- Напиши функцию `extract_ngrams(tokens, n)` → список n-грамм
- Построй TF-IDF поверх unigrams + bigrams
- Проверь, что матрица совпадает с `TfidfVectorizer(ngram_range=(1, 2))`

**Шаг 2** — обучи 4 модели и сравни в таблице:
- Ручной unigram, sklearn unigram, ручной bigram, sklearn bigram


In [ ]:
# TODO: сгенерируй intents_small.csv, если файла нет
# Подсказка: используй по несколько шаблонов на каждый intent


In [ ]:
# TODO: Шаг 1 — реализуй извлечение n-грамм вручную
# Напиши extract_ngrams(tokens, n) -> List[str]:
#   extract_ngrams(["хочу", "отменить", "заказ"], 2) -> ["хочу отменить", "отменить заказ"]
#
# Затем построй класс ManualNgramTfidf с ngram_range=(1, 2):
#   1. Токенизируй текст
#   2. Извлеки все unigrams + bigrams
#   3. Примени TF-IDF поверх них
#
# Проверь совпадение с TfidfVectorizer(ngram_range=(1, 2))

In [ ]:
# TODO: Шаг 2 — обучи 4 модели и сравни accuracy
# TODO: ручной unigram TF-IDF + LogReg
# TODO: sklearn unigram TF-IDF + LogReg
# TODO: ручной unigram+bigram TF-IDF + LogReg
# TODO: sklearn unigram+bigram TF-IDF + LogReg
# TODO: выведи таблицу accuracy всех 4 моделей
# TODO: интерпретируй, почему биграммы помогают или не помогают


In [ ]:
# ---------- Проверка твоего решения (задача 3) ----------
# Ожидаемые переменные:
#   acc_uni — accuracy модели с unigrams
#   acc_bi  — accuracy модели с bigrams
#   vec_uni / vec_bi — обученные TfidfVectorizer
#   X_train_uni, X_train_bi — матрицы признаков

try:
    assert acc_uni is not None and acc_bi is not None
except NameError:
    raise AssertionError(
        "Не найдены переменные acc_uni и/или acc_bi. "
        "Обучи обе модели и посчитай accuracy для каждой."
    )

assert acc_uni > 0.55, (
    f"Unigram accuracy = {acc_uni:.4f} — слишком низкая. "
    "Проверь, что vectorizer обучен на train и stratify задан в split."
)
assert acc_bi > 0.55, (
    f"Bigram accuracy = {acc_bi:.4f} — слишком низкая. "
    "Проверь ngram_range=(1, 2)."
)

try:
    assert X_train_bi.shape[1] > X_train_uni.shape[1], (
        f"Bigram-модель имеет {X_train_bi.shape[1]} признаков, "
        f"unigram — {X_train_uni.shape[1]}. Биграммов должно быть больше. "
        "Проверь ngram_range."
    )
    print(f"  Unigram features: {X_train_uni.shape[1]}, Bigram features: {X_train_bi.shape[1]}")
except NameError:
    print("  (X_train_uni / X_train_bi не найдены, проверка размерности пропущена)")

print(f"Задача 3: unigram={acc_uni:.4f}, bigram={acc_bi:.4f} — проверка пройдена ✓")

### Если проверка не прошла — частые ошибки задачи 3

| Ошибка | Что происходит | Как исправить |
|--------|---------------|---------------|
| `ngram_range=(2, 2)` вместо `(1, 2)` | Модель видит только биграммы, теряет отдельные слова → падение accuracy | Используй `(1, 2)` — unigrams + bigrams вместе |
| Слишком высокие n-граммы `(1, 5)` | Огромное кол-во редких признаков → переобучение на train | Для коротких текстов достаточно `(1, 2)` или `(1, 3)` |
| Разный `random_state` при сравнении | Сравниваешь не методы, а случайные разбиения → неверные выводы | Одинаковый `random_state` в обоих split |
| Нет `stratify` в мультиклассовой задаче | Некоторые классы могут отсутствовать в тесте | `train_test_split(..., stratify=y)` |

## Задача 4: Word2Vec для классификации новостей


Что такое Word2Vec

Все предыдущие методы (BoW, TF-IDF, n-граммы) дают **разреженные** векторы: размерность = размер словаря, большинство элементов = 0. Слова «матч» и «чемпионат» в этих представлениях никак не связаны — это просто два разных столбца.

Word2Vec учит **плотные** (dense) векторы слов по принципу: *слова, встречающиеся в похожих контекстах, должны иметь похожие векторы.*

**Skip-gram** — одна из двух архитектур Word2Vec. Из центрального слова предсказываем контекстные:

```
текст:   [команда] [выиграла] [матч] [сегодня]
                       ↑ центральное (window=1)
          ← контекст →   ← контекст →
```

Пары для обучения: `(выиграла, команда)`, `(выиграла, матч)`.

Модель — два слоя: embedding (W_in) и projection (W_out). Минимизируется cross-entropy: правильное контекстное слово должно получить высокую вероятность.

**Вектор документа** — среднее эмбеддингов его слов. Это работает, потому что слова одной темы (спорт, бизнес) кластеризуются в пространстве эмбеддингов, и среднее по кластеру — это его центроид, а не «ничто».

Есть короткие новости по темам: `sport`, `business`, `politics`, `tech`.

**Шаг 1** — реализуй Skip-Gram вручную:
- Напиши функцию `build_skipgram_pairs(docs, window)` → список пар (center, context)
- Реализуй упрощённый Skip-Gram: два слоя W_in и W_out, softmax, SGD
- Обучи на нескольких эпохах и покажи most_similar через cosine similarity

**Шаг 2** — обучи gensim Word2Vec и сравни:
- Какие слова gensim считает похожими vs ручная модель
- Accuracy классификатора на ручных эмбеддингах vs gensim


In [ ]:
# TODO: сгенерируй news_small.csv, если файла нет
# Подсказка: используй тематические шаблоны по 3-4 слова


In [ ]:
# TODO: Шаг 1 — реализуй Skip-Gram вручную
# 1. build_skipgram_pairs(tokenized_docs, window=2) -> [(center, context), ...]
# 2. SimpleSkipGram: два слоя W_in (vocab × dim), W_out (dim × vocab)
#    - softmax по всему словарю (словарь маленький, это ОК)
#    - cross-entropy loss
#    - градиентный спуск
# 3. Обучи 10-15 эпох и выведи most_similar для нескольких слов
#
# Подсказка: cosine_similarity из sklearn.metrics.pairwise

In [ ]:
# TODO: Шаг 2 — обучи gensim Word2Vec и сравни
# TODO: токенизируй тексты
# TODO: обучи gensim Word2Vec (sg=1, vector_size=50, window=3, epochs=50)
# TODO: выведи most_similar для тех же слов, что и в ручной модели
# TODO: реализуй mean_embedding (среднее эмбеддингов слов документа)
# TODO: обучи LogisticRegression на обеих версиях эмбеддингов (ручная vs gensim)
# TODO: сравни accuracy


In [ ]:
# ---------- Проверка твоего решения (задача 4) ----------
# Ожидаемые переменные:
#   w2v        — обученная модель Word2Vec
#   X_train, X_test — numpy-массивы эмбеддингов документов
#   y_train, y_test — метки
#   acc_w2v или pred_news — accuracy / предсказания

try:
    _acc = acc_w2v
except NameError:
    try:
        _pred = model_news.predict(X_test)
        _acc = accuracy_score(y_test, _pred)
    except NameError:
        raise AssertionError(
            "Не найдены переменные acc_w2v или (model_news, X_test, y_test). "
            "Обучи Word2Vec, получи document embeddings и обучи классификатор."
        )

assert _acc > 0.45, (
    f"Accuracy = {_acc:.4f} — слишком низкая. Возможные причины:\n"
    "  1) OOV-слова не обрабатываются (KeyError или нулевые векторы)\n"
    "  2) Слишком мало эпох обучения Word2Vec\n"
    "  3) Используется сумма вместо среднего для document embedding"
)

try:
    assert X_train.shape[1] == w2v.vector_size, (
        f"Размерность эмбеддингов ({X_train.shape[1]}) != vector_size ({w2v.vector_size}). "
        "Проверь функцию усреднения эмбеддингов."
    )
    _zero = np.all(X_train == 0, axis=1).sum()
    if _zero > 0:
        print(f"  ВНИМАНИЕ: {_zero} документ(ов) с нулевым вектором (все слова OOV).")
except NameError:
    pass

print(f"Задача 4: accuracy = {_acc:.4f} — проверка пройдена ✓")

### Если проверка не прошла — частые ошибки задачи 4

| Ошибка | Что происходит | Как исправить |
|--------|---------------|---------------|
| Нет проверки `if t in model.wv` | `KeyError` при встрече OOV-слова | Фильтруй: `[model.wv[t] for t in tokens if t in model.wv]` |
| `np.sum` вместо `np.mean` | Длинные тексты получают непропорционально большие векторы | Используй `np.mean(vecs, axis=0)` |
| Не возвращать нулевой вектор при пустом списке | `np.mean([])` → warning или NaN → модель падает | `if not vecs: return np.zeros(vector_size)` |
| `vector_size` слишком мал (5–10) | Мало информации для 4 тем → accuracy ~30% | 50 — хороший выбор для маленького корпуса |
| Word2Vec обучен на всех данных | Data leakage через контексты тестовых слов | Обучай Word2Vec только на train-токенах |

## Задача 5: fastText для language identification


Что такое fastText и зачем нужны subwords

Word2Vec учит вектор для каждого **целого слова**. Если слово не встречалось при обучении (OOV) — вектора нет. А ещё Word2Vec не видит внутреннюю структуру слова: «играть», «игрок», «игра» — это для него три совершенно разных объекта.

fastText решает обе проблемы через **символьные n-граммы (subwords)**. К слову добавляются символы-границы `<` (начало) и `>` (конец), затем вырезаются все подстроки длиной от `minn` до `maxn`:

```
«игрок» → «<игрок>»   (7 символов)

minn=3, maxn=5 → скользящее окно по символам:
  длина 3:  <иг  игр  гро  рок  ок>
  длина 4:  <игр  игро  грок  рок>
  длина 5:  <игро  игрок  грок>
```

Границы `<>` нужны, чтобы модель отличала начало слова от середины: `<иг` (приставка) ≠ `иг` внутри другого слова.

Вектор слова = сумма векторов всех его subwords + вектор целого слова. Это даёт:

- **Морфологическую устойчивость**: «играть» и «игрок» делят подстроку `игр` → их векторы частично совпадают.
- **Обработку OOV**: даже если целое слово не встречалось, его подстроки могли встречаться в других словах → модель всё равно даст осмысленный вектор.
- **Устойчивость к опечаткам**: «delivry» вместо «delivery» — большинство subwords совпадают.

Для определения языка это идеально: в каждом языке свои характерные сочетания букв (`sch` — немецкий, `tion` — английский, `ción` — испанский), и fastText их ловит автоматически через subwords.

Параметры `minn` и `maxn` задают диапазон длин символьных n-грамм, `wordNgrams` — n-граммы на уровне слов (биграммы фраз).

Есть короткие фразы на нескольких языках.
Нужно определить язык.

Подумай:
- почему fastText особенно хорош на коротких текстах;
- зачем ему subword information;
- как подготовить строки формата `__label__class text`.


In [ ]:
# TODO: сгенерируй lang_small.csv, если файла нет
# Подсказка: можно взять ru / en / es / de и добавить немного опечаток


In [ ]:
# TODO: разбей данные на train/test
# TODO: сохрани train/test в формате fastText
# TODO: обучи fasttext.train_supervised
# TODO: получи предсказания и выведи accuracy + classification_report


In [ ]:
# ---------- Проверка твоего решения (задача 5) ----------
# Ожидаемые переменные:
#   acc_ft или pred_lang — accuracy / предсказания
#   test_lang            — тестовый DataFrame с колонкой 'label'
#   ft_model             — обученная fastText модель

try:
    _acc = acc_ft
except NameError:
    try:
        _acc = accuracy_score(test_lang["label"], pred_lang)
    except NameError:
        raise AssertionError(
            "Не найдены переменные acc_ft или (pred_lang, test_lang). "
            "Обучи fastText и получи предсказания."
        )

assert _acc > 0.65, (
    f"Accuracy = {_acc:.4f} — слишком низкая. Возможные причины:\n"
    "  1) Неправильный формат файла (нужно __label__<class> <text>)\n"
    "  2) Не удалён __label__ из предсказаний при сравнении\n"
    "  3) Слишком мало эпох или низкий lr"
)

try:
    _pred_set = set(pred_lang)
    _true_set = set(test_lang["label"])
    _unexpected = _pred_set - _true_set
    if _unexpected:
        print(f"  ВНИМАНИЕ: предсказаны метки, которых нет в данных: {_unexpected}")
        print("  Вероятно, __label__ не удалён из предсказаний.")
except NameError:
    pass

print(f"Задача 5: accuracy = {_acc:.4f} — проверка пройдена ✓")

### Если проверка не прошла — частые ошибки задачи 5

| Ошибка | Что происходит | Как исправить |
|--------|---------------|---------------|
| Формат файла `text label` вместо `__label__class text` | fastText не распознаёт метки → accuracy ~0 | `f"__label__{label} {text}\n"` |
| Не удалён `__label__` из предсказаний | `"__label__ru" != "ru"` → accuracy = 0 | `pred.replace("__label__", "")` |
| Мало эпох (epoch=5) и низкий lr (0.1) | Модель не успевает обучиться на маленьком датасете | `epoch=25–50, lr=0.5–1.0` |
| Не указаны `minn`/`maxn` | fastText не использует subword-информацию → теряет главное преимущество | `minn=2, maxn=5` |
| Не указан `wordNgrams=2` | Модель не видит фразы вроде "como estas" → ниже accuracy | `wordNgrams=2` |